# 02.2 CNN Basics

This notebook introduces Convolutional Neural Networks. A convolution layer reads local spatial neighborhoods instead of flattening the whole image immediately. That lets the model learn small visual patterns such as edges or corners and reuse the same pattern detector across different positions.

The main skills are reading `(N, C, H, W)` shapes, predicting how channels change, and understanding how kernel size, stride, padding, and pooling affect spatial dimensions.

## Learning Goals

After this notebook, you should be able to:

1. Understand the input-output structure of `Conv2d`.
2. Explain how shapes change after convolution.
3. Understand the roles of `stride` and `padding`.
4. Understand pooling layers.
5. Read the shape flow of a small CNN.
6. Prepare structurally for later image classification.

In [ ]:
import torch
import torch.nn as nn

## Inputs and Outputs of `Conv2d`

`Conv2d` expects input shaped `(N, C, H, W)`: batch size, input channels, height, and width. The output keeps the same batch size, but the channel count becomes `out_channels`, because the layer learns that many filters. The output height and width depend on kernel size, stride, and padding.

When reading CNN code, track the channel dimension and the spatial dimensions separately. Channels usually change because of convolution filters; height and width change because of padding, stride, or pooling.

In [ ]:
x = torch.randn(4, 1, 8, 8)
conv = nn.Conv2d(in_channels=1, out_channels=3, kernel_size=3, stride=1, padding=1)
y = conv(x)

print("x.shape =", x.shape)
print("y.shape =", y.shape)

The output shape is `(4, 3, 8, 8)`. The batch size stays 4 because convolution processes each image independently. The channel count becomes 3 because `out_channels=3`. The height and width stay 8 because `kernel_size=3`, `padding=1`, and `stride=1` preserve spatial size.

## `kernel_size`, `stride`, and `padding`

`kernel_size` is the spatial size of the patch the convolution looks at each time. `stride` controls how far the kernel moves between positions. Larger stride usually makes the output spatial size smaller. `padding` adds border values around the image so the kernel can cover edge positions; with `kernel_size=3`, `padding=1`, and `stride=1`, the height and width stay the same.

These three arguments determine whether a CNN keeps, shrinks, or aggressively compresses spatial information.

In [ ]:
x = torch.randn(1, 1, 8, 8)

conv_a = nn.Conv2d(1, 4, kernel_size=3, stride=1, padding=0)
conv_b = nn.Conv2d(1, 4, kernel_size=3, stride=1, padding=1)
conv_c = nn.Conv2d(1, 4, kernel_size=3, stride=2, padding=1)

print("input shape =", x.shape)
print("padding=0, stride=1 ->", conv_a(x).shape)
print("padding=1, stride=1 ->", conv_b(x).shape)
print("padding=1, stride=2 ->", conv_c(x).shape)

## Number of Parameters

Convolution layers also contain learnable parameters.

For `Conv2d(in_channels=C_in, out_channels=C_out, kernel_size=k)`:

- number of weight parameters: `C_out * C_in * k * k`
- add another `C_out` if bias is used

In [ ]:
conv = nn.Conv2d(1, 3, kernel_size=3)

num_params = sum(p.numel() for p in conv.parameters())
print("num_params =", num_params)

# weights = 3 * 1 * 3 * 3 = 27
# bias = 3
# total = 30

In [ ]:
# Exercise 1
#
# Compute the parameter count for:
# nn.Conv2d(in_channels=3, out_channels=8, kernel_size=5)
#
# Manual calculation:
# - Each output channel has a 3 * 5 * 5 weight block.
# - There are 8 output channels.
# - If bias=True, there is one bias value per output channel.
#
# Write the manual calculation as a comment, then create conv_ex and verify the
# count with sum(p.numel() for p in conv_ex.parameters()).

# conv_ex =
# print(sum(p.numel() for p in conv_ex.parameters()))

In [ ]:
# Exercise 1 Reference Solution

conv_ex = nn.Conv2d(in_channels=3, out_channels=8, kernel_size=5)
print(sum(p.numel() for p in conv_ex.parameters()))

# weights = 8 * 3 * 5 * 5 = 600
# bias = 8
# total = 608

## Pooling Layers

Pooling is commonly used to reduce spatial dimensions.

The most common one is:

- `MaxPool2d`

Its intuition is: take the maximum value within a small local window.


In [ ]:
x = torch.randn(2, 4, 8, 8)
pool = nn.MaxPool2d(kernel_size=2, stride=2)
y = pool(x)

print("x.shape =", x.shape)
print("y.shape =", y.shape)

Here the spatial size changes from `8x8` to `4x4` because a `2x2` pooling window with `stride=2` halves both height and width.


## Shape Flow in a Minimal CNN

When reading CNN code, the most important thing is to keep tracking the shape.


In [ ]:
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2)
        self.fc = nn.Linear(16 * 2 * 2, 10)

    def forward(self, x):
        print("input:", x.shape)
        x = self.conv1(x)
        print("after conv1:", x.shape)
        x = self.relu1(x)
        x = self.pool1(x)
        print("after pool1:", x.shape)
        x = self.conv2(x)
        print("after conv2:", x.shape)
        x = self.relu2(x)
        x = self.pool2(x)
        print("after pool2:", x.shape)
        x = x.flatten(start_dim=1)
        print("after flatten:", x.shape)
        x = self.fc(x)
        print("after fc:", x.shape)
        return x


model = TinyCNN()
dummy = torch.randn(4, 1, 8, 8)
out = model(dummy)

Pay special attention to the shape chain in the CNN above. The input starts as `(4, 1, 8, 8)`, meaning 4 grayscale images. The first convolution changes channels from 1 to 8 while preserving spatial size. Max pooling then shrinks height and width from 8 to 4. The second convolution changes channels from 8 to 16. The second pooling shrinks the spatial size again. Finally, flattening turns channels and spatial dimensions into one feature dimension before the linear classifier outputs 10 class scores.

This kind of shape tracing is the basic skill required to read CNN code.

In [ ]:
# Exercise 2
#
# Trace the output shape.
#
# Input shape:
# - x starts as (batch, 1, 8, 8).
#
# Layers:
# 1. Conv2d(1, 4, kernel_size=3, padding=1)
# 2. MaxPool2d(2)
#
# What to do:
# - Compute the output shape by hand first.
# - Then create x, conv, and pool to verify it with code.
# - Remember that padding=1 preserves 8x8 for the convolution, and MaxPool2d(2)
#   halves height and width.

# x =
# conv =
# pool =
# y =
# print(y.shape)

In [ ]:
# Exercise 2 Reference Solution

x = torch.randn(5, 1, 8, 8)
conv = nn.Conv2d(1, 4, kernel_size=3, padding=1)
pool = nn.MaxPool2d(2)
y = pool(conv(x))
print(y.shape)

# shape: (5, 4, 4, 4)

## `Flatten` and Linear Layers

The output of convolution layers is still usually a 4D tensor.

If a linear layer comes next, we usually flatten first.


In [ ]:
x = torch.randn(3, 16, 2, 2)
x_flat = x.flatten(start_dim=1)

print("x.shape =", x.shape)
print("x_flat.shape =", x_flat.shape)

`16 * 2 * 2 = 64`, so after flattening the shape becomes `(batch_size, 64)`.


In [ ]:
# Exercise 3
#
# A layer output has shape (7, 12, 3, 3).
#
# Question:
# - What is the shape after flatten(start_dim=1)?
#
# Explain your answer. The first dimension is batch size and should stay 7. The
# remaining dimensions should be multiplied into one feature dimension.

Exercise 3 Reference Answer

`12 * 3 * 3 = 108`, so the shape becomes `(7, 108)`.

## Summary

The most important outcome of this notebook is building CNN shape intuition.

You should now be able to answer:

1. What do the input and output shapes of `Conv2d` usually look like?
2. Why does `out_channels` change the number of feature-map channels?
3. Why do pooling layers often reduce height and width?
4. Why do we often flatten convolution outputs before linear layers?

Suggested next step:

- Move to the CNN classification notebook and put these structures into a real classification task.